In [ ]:
import math

def entropy(labels):
    if len(labels) == 0:
        return 0

    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1

    result = 0
    total = len(labels)

    for count in counts.values():
        p = count / total
        result -= p * math.log2(p)

    return result

labels1 = ["Pass", "Pass", "Pass", "Pass"]
labels2 = ["Pass", "Pass", "Fail", "Fail"]
labels3 = ["Pass", "Fail", "Pass", "Fail"]

print(entropy(labels1))
print(entropy(labels2))
print(entropy(labels3))

In [ ]:
dataset = [
    {"Attendance": "High", "Study": "High", "Assignment": "Yes", "Label": "Pass"},
    {"Attendance": "High", "Study": "Low", "Assignment": "Yes", "Label": "Pass"},
    {"Attendance": "High", "Study": "Low", "Assignment": "No", "Label": "Pass"},
    {"Attendance": "Low", "Study": "High", "Assignment": "Yes", "Label": "Pass"},
    {"Attendance": "Low", "Study": "Low", "Assignment": "No", "Label": "Fail"},
    {"Attendance": "Low", "Study": "Low", "Assignment": "Yes", "Label": "Fail"},
    {"Attendance": "High", "Study": "High", "Assignment": "No", "Label": "Pass"},
    {"Attendance": "Low", "Study": "High", "Assignment": "No", "Label": "Fail"}
]

features = ["Attendance", "Study", "Assignment"]

def conditional_entropy(dataset, feature, label_col="Label"):
    total = len(dataset)
    result = 0
    values = set(row[feature] for row in dataset)

    for value in values:
        subset = [row[label_col] for row in dataset if row[feature] == value]
        result += len(subset) / total * entropy(subset)

    return result

def mutual_information(dataset, feature, label_col="Label"):
    labels = [row[label_col] for row in dataset]
    return entropy(labels) - conditional_entropy(dataset, feature, label_col)

for feature in features:
    print(feature, mutual_information(dataset, feature))

In [ ]:
class Node:
    def __init__(self, node_type, label=None, feature=None):
        self.type = node_type
        self.label = label
        self.feature = feature
        self.branches = {}

def predict(tree, x):
    current = tree

    while current.type != "leaf":
        feature = current.feature
        value = x[feature]
        current = current.branches[value]

    return current.label

fail_leaf = Node("leaf", label="Fail")
pass_leaf = Node("leaf", label="Pass")

study_node = Node("internal", feature="Study")
study_node.branches["Low"] = fail_leaf
study_node.branches["High"] = pass_leaf

root = Node("internal", feature="Attendance")
root.branches["Low"] = fail_leaf
root.branches["High"] = study_node

students = [
    {"Attendance": "High", "Study": "High"},
    {"Attendance": "High", "Study": "Low"},
    {"Attendance": "Low", "Study": "High"}
]

for student in students:
    print(predict(root, student))

In [ ]:
def majority_label(dataset, label_col="Label"):
    counts = {}
    for row in dataset:
        counts[row[label_col]] = counts.get(row[label_col], 0) + 1
    return max(counts, key=counts.get)

def train(dataset, features, label_col="Label"):
    labels = [row[label_col] for row in dataset]

    if len(set(labels)) == 1:
        return Node("leaf", label=labels[0])

    if len(features) == 0:
        return Node("leaf", label=majority_label(dataset, label_col))

    best_feature = features[0]
    best_mi = mutual_information(dataset, best_feature, label_col)

    for feature in features[1:]:
        mi = mutual_information(dataset, feature, label_col)
        if mi > best_mi:
            best_mi = mi
            best_feature = feature

    node = Node("internal", feature=best_feature)
    values = set(row[best_feature] for row in dataset)
    remaining_features = [f for f in features if f != best_feature]

    for value in values:
        subset = [row for row in dataset if row[best_feature] == value]
        node.branches[value] = train(subset, remaining_features, label_col)

    return node

tree = train(dataset, features)
print(tree.feature)

In [ ]:
new_students = [
    {"Attendance": "High", "Study": "High", "Assignment": "Yes"},
    {"Attendance": "High", "Study": "Low", "Assignment": "No"},
    {"Attendance": "Low", "Study": "High", "Assignment": "Yes"},
    {"Attendance": "Low", "Study": "Low", "Assignment": "No"},
    {"Attendance": "High", "Study": "Low", "Assignment": "Yes"}
]

for student in new_students:
    print(student, predict(tree, student))